## **MLlib 튜토리얼**

In [61]:
# [+] SparkSession 설정
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("SparkApp") \
    .getOrCreate()

### **MLlib 예제: Logistic Regression**
+ ```pyspark.ml.linalg```: 선형대수(linear algebra) 관련 패키지
    + ```Vectors```: 벡터 데이터 타입
+ ```pyspark.ml.classification```: 분류 모델 관련 패키지
    + ```LogisticRegression```: 가장 기본적인 분류 모델

In [62]:
# Vectors, LogisticRegression 임포트
from pyspark.ml.linalg import Vectors
from pyspark.ml.classification import LogisticRegression

**데이터 정의: 레이블과 특징**
+ ```dense()```: 모든 값을 저장하는 벡터 생성
+ ```sparse()```: 0이 아닌 값만 저장하는 벡터 생성

In [63]:
# 훈련 데이터: (label, features)
train_data = [
    (1.0, Vectors.dense([0.0, 1.1, 0.1])),
    (0.0, Vectors.dense([2.0, 1.0, -1.0])),
    (0.0, Vectors.dense([2.0, 1.3, 1.0])),
    (1.0, Vectors.dense([0.0, 1.2, -0.5])),
]

In [64]:
# [+] 스키마 정의
schema = ['label', 'features']

In [65]:
# [+] 데이터프레임 생성
train_df = spark.createDataFrame(train_data, schema)

In [66]:
# 데이터프레임 출력 
train_df.show()

+-----+--------------+
|label|      features|
+-----+--------------+
|  1.0| [0.0,1.1,0.1]|
|  0.0|[2.0,1.0,-1.0]|
|  0.0| [2.0,1.3,1.0]|
|  1.0|[0.0,1.2,-0.5]|
+-----+--------------+



**LogisticRegression (Estimator) 생성**
+ ```maxIter```: 최대 학습 횟수 매개변수
+ ```regParam```: 정규화 매개변수

In [67]:
# [+] Estimator(학습 알고리즘) 객체 생성
lr = LogisticRegression(maxIter=30, regParam=0.01)

In [68]:
# [+] 모델 학습
model = lr.fit(train_data)

AttributeError: 'list' object has no attribute '_jdf'

In [69]:
# 모델 출력
model

PipelineModel_1ab194b7188d

In [70]:
# 테스트셋
test_data = [
    (1.0, Vectors.dense([-1.0, 1.5, 1.3])),
    (0.0, Vectors.dense([3.0, 2.0, -0.1])),
    (1.0, Vectors.dense([0.0, 2.2, -1.5])),
]

In [71]:
# [+] 데이터프레임 생성
test_df = spark.createDataFrame(test_data, schema)

In [72]:
# [+] 모델 예측
predictions = model.transform(test_df)

IllegalArgumentException: text does not exist. Available: label, features

In [73]:
# 예측결과 출력
predictions.show()

+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+
| id|              text|               words|            features|       rawPrediction|         probability|prediction|
+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+
|  4|       spark i j k|    [spark, i, j, k]|(262144,[19036,68...|[0.52882855227968...|[0.62920984896684...|       0.0|
|  5|             l m n|           [l, m, n]|(262144,[1303,526...|[4.16914139534005...|[0.98477000676230...|       0.0|
|  6|spark hadoop spark|[spark, hadoop, s...|(262144,[173558,1...|[-1.8649814141188...|[0.13412348342566...|       1.0|
|  7|     apache hadoop|    [apache, hadoop]|(262144,[68303,19...|[5.41564427200184...|[0.99557321143985...|       0.0|
+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+



### **LogisticRegression 파이프라인 구현하기**
+ 예측 문제: Spark에 대한 텍스트 인지 아닌지를 분류
+ ```Pipeline```: 머신러닝 과정에 대한 파이프라인 정의
+ ```HashingTF```: 컬럼을 용어빈도(term-frequency) 컬럼으로 변환하는 Transformer
+ ```Tokenizer```: 컬럼의 텍스트를 여러 개의 단어로 분할하는 Transformer

In [74]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, Tokenizer

In [75]:
# 훈련 데이터를 데이터프레임으로 생성
train_df = spark.createDataFrame([
    (0, "a b c d e spark", 1.0),
    (1, "b d", 0.0),
    (2, "spark f g h", 1.0),
    (3, "hadoop mapreduce", 0.0)
], ["id", "text", "label"])

In [76]:
train_df.show()

+---+----------------+-----+
| id|            text|label|
+---+----------------+-----+
|  0| a b c d e spark|  1.0|
|  1|             b d|  0.0|
|  2|     spark f g h|  1.0|
|  3|hadoop mapreduce|  0.0|
+---+----------------+-----+



**머신러닝 수행 과정**

Tokenizer ${\rightarrow}$ HashingTF ${\rightarrow}$ LogisticRegression

In [77]:
# [+] Tokenizer 객체 생성
tokenizer = Tokenizer(inputCol='text',outputCol='words')

In [78]:
# 토크나이징 수행
df_tok = tokenizer.transform(train_df)
df_tok.show()

+---+----------------+-----+--------------------+
| id|            text|label|               words|
+---+----------------+-----+--------------------+
|  0| a b c d e spark|  1.0|[a, b, c, d, e, s...|
|  1|             b d|  0.0|              [b, d]|
|  2|     spark f g h|  1.0|    [spark, f, g, h]|
|  3|hadoop mapreduce|  0.0| [hadoop, mapreduce]|
+---+----------------+-----+--------------------+



In [79]:
# [+] HashingTF 객체 생성
hashingTF = HashingTF(
    inputCol=tokenizer.getOutputCol(), 
    outputCol='features'
)

In [80]:
# 단어빈도 벡터로 변환
df_tf = hashingTF.transform(df_tok)
df_tf.show()

+---+----------------+-----+--------------------+--------------------+
| id|            text|label|               words|            features|
+---+----------------+-----+--------------------+--------------------+
|  0| a b c d e spark|  1.0|[a, b, c, d, e, s...|(262144,[74920,89...|
|  1|             b d|  0.0|              [b, d]|(262144,[89530,14...|
|  2|     spark f g h|  1.0|    [spark, f, g, h]|(262144,[36803,17...|
|  3|hadoop mapreduce|  0.0| [hadoop, mapreduce]|(262144,[132966,1...|
+---+----------------+-----+--------------------+--------------------+



**LogisticRegression 객체 생성**
+ ```maxIter```: 10
+ ```regParam```: 0.001

In [81]:
# [+] Estimator 객체 생성
lr = LogisticRegression(maxIter=10, regParam=0.001)

In [82]:
# 모델 학습
model = lr.fit(df_tf)

In [83]:
# 모델 학습 결과(특징별 계수인터셉트)
print(model.coefficients)
print(model.intercept)

(262144,[36803,74920,89530,107107,132966,148981,167694,173558,198017,209078,228158],[2.2140795257626475,2.639670931651186,-0.7256824868256397,2.639670931651186,-1.2465028766617947,-0.7256824868256397,2.639670931651186,3.640312843060374,-1.2465028766617947,2.2140795257626475,2.2140795257626475])
-4.169141395340055


**머신러닝 파이프라인 생성 및 실행**\
```Pipeline.fit()```은 다음의 절차들을 일괄적으로 수행
1. 토크나이징(Tokenizer): text ${\rightarrow}$ words
2. 단어빈도(Term-Frequency) 벡터로 변환: words ${\rightarrow}$ features
3. 로지스틱스 회귀모델(LogisticRegression) 학습

In [84]:
# [+] Pipeline 객체 생성
pipeline = Pipeline(stages=[tokenizer, hashingTF, lr])

In [85]:
# [+] Pipeline 실행
model = pipeline.fit(train_df)

In [86]:
# 테스트 데이터에 대한 DataFrame 생성
test_df = spark.createDataFrame([
    (4, "spark i j k"),
    (5, "l m n"),
    (6, "spark hadoop spark"),
    (7, "apache hadoop")
], ["id", "text"])

In [87]:
# [+] 테스트 데이터에 대한 예측
predictions = model.transform(test_df)

In [88]:
predictions.show()

+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+
| id|              text|               words|            features|       rawPrediction|         probability|prediction|
+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+
|  4|       spark i j k|    [spark, i, j, k]|(262144,[19036,68...|[0.52882855227968...|[0.62920984896684...|       0.0|
|  5|             l m n|           [l, m, n]|(262144,[1303,526...|[4.16914139534005...|[0.98477000676230...|       0.0|
|  6|spark hadoop spark|[spark, hadoop, s...|(262144,[173558,1...|[-1.8649814141188...|[0.13412348342566...|       1.0|
|  7|     apache hadoop|    [apache, hadoop]|(262144,[68303,19...|[5.41564427200184...|[0.99557321143985...|       0.0|
+---+------------------+--------------------+--------------------+--------------------+--------------------+----------+

